In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [ ]:
# =================================================================
# 1. DATA LOADING
# =================================================================
# Ensure 'ai_job_impact.csv' is in the same folder as this script
df = pd.read_csv('ai_job_impact.csv')


In [ ]:
# =================================================================
# 2. PANDAS & NUMPY OPERATIONS (Data Wrangling)
# =================================================================
# Create new metrics
df['Salary_Diff'] = df['Salary_After_AI'] - df['Salary_Before_AI']
df['Salary_Change_Pct'] = (df['Salary_Diff'] / df['Salary_Before_AI']) * 100

In [ ]:
# Logic-based categorization using NumPy
df['Salary_Level'] = np.where(df['Salary_After_AI'] > df['Salary_After_AI'].median(), 'High', 'Low')

# Complex Grouping
industry_performance = df.groupby('Industry').agg({
    'Salary_After_AI': 'mean',
    'Productivity_Change_%': 'mean',
    'Job_Satisfaction': 'mean'
}).sort_values(by='Productivity_Change_%', ascending=False)

print("--- Industry Performance Summary ---")
print(industry_performance.head())


In [ ]:
# =================================================================
# 3. STATISTICAL ANALYSIS (SciPy)
# =================================================================
# T-Test: Is the difference in productivity between Remote and On-site significant?
remote = df[df['Remote_Work'] == 'Yes']['Productivity_Change_%']
onsite = df[df['Remote_Work'] == 'No']['Productivity_Change_%']
t_stat, p_val = stats.ttest_ind(remote, onsite)

print(f"\nRemote vs On-site Productivity P-Value: {p_val:.4f}")

In [ ]:
# =================================================================
# 4. ML PREPROCESSING (Scikit-Learn)
# =================================================================
# Standardizing numerical data
scaler = StandardScaler()
df['Age_Normalized'] = scaler.fit_transform(df[['Age']])

# Encoding Categorical Text into Numbers
le = LabelEncoder()
df['Industry_Encoded'] = le.fit_transform(df['Industry'])

In [ ]:
# =================================================================
# 5. DATA VISUALIZATION (Matplotlib & Seaborn)
# =================================================================
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Correlation Heatmap
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, cmap='coolwarm', ax=axes[0,0])
axes[0,0].set_title('Feature Correlation Matrix')

# Plot 2: Salary Change by AI Adoption Level
sns.boxplot(x='AI_Adoption_Level', y='Salary_Change_Pct', data=df, palette='Set2', ax=axes[0,1])
axes[0,1].set_title('Salary Change % vs AI Adoption')



In [ ]:
# Plot 3: Productivity Change Density by Gender
sns.kdeplot(data=df, x='Productivity_Change_%', hue='Gender', fill=True, ax=axes[1,0])
axes[1,0].set_title('Productivity Distribution by Gender')

# Plot 4: Automation Risk vs Job Status
sns.countplot(data=df, x='Automation_Risk', hue='Job_Status', palette='magma', ax=axes[1,1])
axes[1,1].set_title('Job Status impact by Automation Risk')

plt.tight_layout()
plt.show()

In [ ]:
# =================================================================
# 6. FINAL EXPORT
# =================================================================
df.to_csv('ai_job_impact_final_results.csv', index=False)
print("\nSuccess! Results saved to 'ai_job_impact_final_results.csv'")